# RoBERTa Model (Macro F1: 0.66)

**Model:** roberta-base  
**Validation Macro F1:** 0.66 (scored by codabench)

In [ ]:
#YOOOO this one achieved 0.66 f1 score!!!!
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import gc
import matplotlib.pyplot as plt
import seaborn as sns

# ====================== CONFIG ======================
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
EPOCHS = 8
LEARNING_RATE = 1e-5
OUTPUT_DIR = "./roberta_v1_fixed"

# Load data
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test_input.csv")

# ====================== IMPROVED INPUT FORMAT ======================
def create_input(row):
    return f"""Question: {row['Post']}

Comment: {row['Comment']}

Is the comment a direct answer to the question, only related to the topic, or completely off-topic?"""

# ====================== LIGHT AUGMENTATION ======================
print("Applying light augmentation on off-topic...")
off_topic_df = train_df[train_df['Label'] == 'off-topic']
augmented_off_topic = []

for idx, row in off_topic_df.iterrows():
    for i in range(3):
        new_row = row.copy()
        words = str(row['Post']).split()
        if len(words) > 8:
            pos1, pos2 = np.random.choice(len(words), 2, replace=False)
            words[pos1], words[pos2] = words[pos2], words[pos1]
            new_row['Post'] = ' '.join(words)
        augmented_off_topic.append(new_row)

augmented_df = pd.concat([train_df, pd.DataFrame(augmented_off_topic)], ignore_index=True)
train_df = augmented_df

print(f"Training size after augmentation: {len(train_df)}")

# Create text
train_df['text'] = train_df.apply(create_input, axis=1)
val_df['text'] = val_df.apply(create_input, axis=1)
test_df['text'] = test_df.apply(create_input, axis=1)

# Label mapping
label2id = {"direct": 0, "related": 1, "off-topic": 2}
id2label = {v: k for k, v in label2id.items()}

train_df['label'] = train_df['Label'].map(label2id)
val_df['label'] = val_df['Label'].map(label2id)

# Datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['text', 'label']])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

print("Tokenizing...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Class weights
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].values
)
class_weights = torch.tensor(class_weights_array, dtype=torch.float32)
print(f"Class weights: {class_weights_array}")

# Custom Weighted Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.to(torch.float32)
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device).type_as(logits)
        )
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Clear memory
torch.cuda.empty_cache()
gc.collect()

# Load model
print(f"\nLoading {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

# Training Arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_train_epochs=EPOCHS,
    weight_decay=0.05,
    warmup_ratio=0.2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
    save_total_limit=3,
    lr_scheduler_type="cosine",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, predictions, average='macro')}

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("\n" + "="*60)
print("🚀 Starting training with improved input format...")
print("="*60 + "\n")

trainer.train()

# ====================== CONFUSION MATRIX (Safe Version) ======================
print("\nGenerating confusion matrix on validation set...")

trainer.model.eval()

with torch.no_grad():
    val_predictions = trainer.predict(tokenized_val)
    val_preds = np.argmax(val_predictions.predictions, axis=-1)
    val_labels = val_predictions.label_ids

# Print matrix
cm = confusion_matrix(val_labels, val_preds)
print("\nValidation Confusion Matrix:")
print(pd.DataFrame(cm, 
                   index=["direct", "related", "off-topic"],
                   columns=["direct", "related", "off-topic"]))

# Plot seaborn version
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["direct", "related", "off-topic"],
            yticklabels=["direct", "related", "off-topic"])
plt.title('Confusion Matrix - RoBERTa-base (Validation Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# ====================== SAVE MODEL & GENERATE TEST PREDICTIONS ======================
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

print("\nGenerating predictions on test set...")
test_dataset = Dataset.from_pandas(test_df[['text']])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

predictions = trainer.predict(tokenized_test)
pred_labels = np.argmax(predictions.predictions, axis=-1)

test_df['Label'] = [id2label[label] for label in pred_labels]
submission = test_df[['ID', 'Label']]
submission.to_csv("predictions.csv", index=False)

print("✅ predictions.csv created successfully!")
print("\nFinal Prediction Distribution:")
print(submission['Label'].value_counts())